<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 7 · Tablas dinámicas que responden preguntas
**Excel Corporativo · 12 Claves** — pivot_table, crosstab y gráficas de análisis · ⏱ 60 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/07_tablas_dinamicas.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-07)

**Al terminar podrás:**
1. Convertir preguntas de negocio en filas, columnas, valores y filtros.
2. Distinguir suma, recuento de líneas y recuento distinto de facturas.
3. Calcular márgenes y participaciones sin errores de agregación.
4. Comparar la venta real con las metas y redactar el hallazgo.

**Cómo usar este cuaderno.** Lee cada explicación y ejecuta la celda de código que sigue con **Shift + Enter**. Las celdas con un título y el código oculto (*Comprobar*, *Ver una solución*) se ejecutan igual; haz doble clic sobre ellas si quieres ver su código. Si algo falla, usa *Entorno de ejecución › Reiniciar y ejecutar todo*.

## Excel y Python, lado a lado

| Tabla dinámica en Excel | En pandas |
|---|---|
| Filas, Columnas, Valores | `pd.pivot_table(df, index=..., columns=..., values=..., aggfunc=...)` |
| Filtro de informe | `df[condición]` antes de resumir |
| Recuento | `aggfunc="count"` (cuenta líneas) |
| Recuento distinto (Modelo de datos) | `aggfunc="nunique"` |
| Mostrar valores como › % del total de fila | `pd.crosstab(..., normalize="index")` |
| Totales generales | `margins=True` |

In [ ]:
# @title Preparar el entorno (ejecuta esta celda primero)
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"   # si clonaste el repositorio, usa los archivos locales

def ruta(nombre):
    """Devuelve la ruta local del archivo si existe; si no, la URL de GitHub."""
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

def pesos(x):
    """Formato de pesos colombianos: $ 1.234.567"""
    return "$ " + f"{x:,.0f}".replace(",", ".")

def millones(x):
    return "$ " + f"{x/1e6:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".") + " M"

def porc(v, signo=False):
    """Porcentaje con coma decimal: 18,9 %"""
    txt = f"{v:+.1%}" if signo else f"{v:.1%}"
    return txt.replace(".", ",").replace("%", " %")

def comprobar(respuesta, esperado, tolerancia=0.0, unidad=""):
    """Compara tu respuesta con el resultado esperado y te da pistas."""
    if respuesta is None:
        print("Aún no has escrito tu respuesta: reemplaza None por tu resultado y vuelve a ejecutar.")
        return
    if isinstance(esperado, str):
        ok = str(respuesta).strip().lower() == esperado.strip().lower()
    else:
        try:
            ok = abs(float(respuesta) - float(esperado)) <= tolerancia
        except (TypeError, ValueError):
            print("Tu respuesta debe ser un número."); return
    if ok:
        print(f"Correcto. {respuesta}{unidad} es el resultado esperado.")
    else:
        print(f"Todavía no. Revisa los filtros (¿quitaste las anuladas?) y el periodo. Tu respuesta: {respuesta}{unidad}")

# Plantilla de gráficos Dr. Bits: separador decimal coma y miles punto, como en Colombia
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=["#4A5AD8", "#0E9BB8", "#D0336F"],
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    yaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    separators=",.", hoverlabel=dict(font_size=13), margin=dict(t=70, l=10, r=20, b=40),
))
pio.templates.default = "plotly_white+drbits"
def dar_aire(fig, maximo, eje="x"):
    """Deja espacio a la derecha (o arriba) para que las etiquetas de las barras no se corten."""
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

MM = "Millones de pesos"   # en español «billón» es 10^12: evitamos la abreviatura B de Plotly
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pd.set_option("display.max_columns", 30)
pd.set_option("styler.format.decimal", ",")      # tablas con estilo: 18,9 %
pd.set_option("styler.format.thousands", ".")
print("Entorno listo. Datos desde:", "archivos locales" if os.path.exists(BASE_LOCAL) else "GitHub")

## Parte A · Armar el modelo
Una tabla dinámica en Excel trabaja sobre una sola tabla, o sobre el Modelo de datos con relaciones. En pandas unimos la venta con sus maestros, con `validate` para que ninguna unión multiplique filas (clave 5).

In [ ]:
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
productos = pd.read_csv(ruta("productos.csv"))
clientes = pd.read_csv(ruta("clientes.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))

df = (ventas
      .merge(productos[["sku", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
print(df.shape)
df.head(3)

### El filtro que casi todos olvidan
Las facturas anuladas no son venta. Si la pregunta es comercial, se excluyen **antes** de resumir. Mira cuánto cambiaría el total si no lo haces.

In [ ]:
validas = df[df["estado"] != "Anulada"]
print("Venta neta con anuladas:", millones(df["subtotal"].sum()))
print("Venta neta sin anuladas:", millones(validas["subtotal"].sum()))
print("Diferencia:             ", millones(df["subtotal"].sum() - validas["subtotal"].sum()))

## Parte B · Seis preguntas, seis tablas dinámicas
Cada pregunta sigue el mismo método: **escribe la pregunta → decide filtros, filas, columnas y valor → resume → grafica → redacta el hallazgo.** Son las mismas preguntas del simulador del tablero: los números deben coincidir.

### Pregunta 1 · ¿Qué ciudad vendió más en 2025?
Filtro: 2025. Filas: ciudad. Valor: suma de subtotal.

In [ ]:
p1 = (validas[validas["anio"] == 2025]
      .pivot_table(index="ciudad", values="subtotal", aggfunc="sum")
      .sort_values("subtotal", ascending=False))
p1["participación"] = p1["subtotal"] / p1["subtotal"].sum()
p1.style.format({"subtotal": pesos, "participación": "{:.1%}"})

In [ ]:
datos = p1.reset_index().sort_values("subtotal")
fig = px.bar(datos, x=datos["subtotal"] / 1e6, y="ciudad", orientation="h", text=datos["subtotal"].map(millones),
             color_discrete_sequence=[AZUL], title="Venta neta 2025 por ciudad")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(xaxis_title=MM, yaxis_title="", height=360)
dar_aire(fig, datos["subtotal"].max() / 1e6)
fig.show()

### Pregunta 2 · ¿Cuánto creció cada ciudad de 2024 a 2025?
Filas: ciudad. Columnas: año. Valor: suma de subtotal. La variación se calcula **después** de resumir.

In [ ]:
p2 = validas.pivot_table(index="ciudad", columns="anio", values="subtotal", aggfunc="sum")
p2["variación"] = p2[2025] / p2[2024] - 1
p2 = p2.sort_values("variación", ascending=False)
p2.style.format({2024: pesos, 2025: pesos, "variación": "{:+.1%}"})

In [ ]:
largo = p2[[2024, 2025]].reset_index().melt(id_vars="ciudad", var_name="año", value_name="subtotal")
largo["año"] = largo["año"].astype(str)
largo["millones"] = largo["subtotal"] / 1e6
fig = px.bar(largo, x="ciudad", y="millones", color="año", barmode="group",
             color_discrete_map={"2024": GRIS, "2025": AZUL},
             category_orders={"ciudad": list(p2.index)},
             title="Venta neta por ciudad, 2024 frente a 2025 (ordenado por crecimiento)")
for i, (ciudad, fila) in enumerate(p2.iterrows()):
    fig.add_annotation(x=ciudad, y=max(fila[2024], fila[2025]) / 1e6, text=porc(fila["variación"], signo=True),
                       showarrow=False, yshift=12, font=dict(size=12, color="#11153A"))
fig.update_layout(yaxis_title=MM, xaxis_title="", legend_title="")
fig.show()

### Pregunta 3 · ¿Qué línea deja más margen porcentual?
Aquí está el error más frecuente con porcentajes. El margen % de un grupo es **Σ margen ÷ Σ venta**. Promediar el margen % de cada línea da el mismo peso a un paquete de lápices que a un lote de portátiles.

In [ ]:
p3 = validas.groupby("categoria").agg(venta=("subtotal", "sum"), margen=("margen_bruto", "sum"))
p3["margen % correcto"] = p3["margen"] / p3["venta"]
p3["promedio de % (incorrecto)"] = (validas.assign(pct=validas["margen_bruto"] / validas["subtotal"])
                                    .groupby("categoria")["pct"].mean())
p3 = p3.sort_values("margen % correcto", ascending=False)
total_ok = validas["margen_bruto"].sum() / validas["subtotal"].sum()
total_mal = (validas["margen_bruto"] / validas["subtotal"]).mean()
print("Empresa: margen correcto", porc(total_ok), "| promedio de porcentajes", porc(total_mal))
p3.style.format({"venta": pesos, "margen": pesos, "margen % correcto": "{:.1%}", "promedio de % (incorrecto)": "{:.1%}"})

In [ ]:
datos = p3.reset_index().sort_values("margen % correcto")
fig = px.bar(datos, x="margen % correcto", y="categoria", orientation="h",
             text=datos["margen % correcto"].map(lambda v: f"{v:.1%}".replace(".", ",")),
             color_discrete_sequence=[TEAL], title="Margen bruto % por línea (Σ margen ÷ Σ venta)")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(xaxis_tickformat=".0%", xaxis_title="", yaxis_title="", height=330)
dar_aire(fig, datos["margen % correcto"].max())
fig.show()

Tecnología es la línea que más vende y la que menos margen deja. Esa tensión entre volumen y rentabilidad es un hallazgo que vale la pena llevar a un informe.

### Pregunta 4 · ¿El e-commerce ganó participación?
Filas: año. Columnas: canal. Mostrar valores como **% del total de fila**. En pandas: `crosstab` con `normalize="index"`.

In [ ]:
p4 = pd.crosstab(validas["anio"], validas["canal"], values=validas["subtotal"], aggfunc="sum", normalize="index")
p4 = p4[["Presencial", "Televentas", "E-commerce"]]
p4.style.format("{:.1%}")

In [ ]:
largo = p4.reset_index().melt(id_vars="anio", var_name="canal", value_name="participación")
largo["anio"] = largo["anio"].astype(str)
fig = px.bar(largo, y="anio", x="participación", color="canal", orientation="h",
             text=largo["participación"].map(lambda v: f"{v:.0%}"),
             color_discrete_sequence=[AZUL, TEAL, MAGENTA],
             title="Participación de cada canal en la venta neta")
fig.update_traces(marker_line_color="white", marker_line_width=2, insidetextanchor="middle")
fig.update_layout(barmode="stack", xaxis_tickformat=".0%", yaxis_title="", xaxis_title="",
                  legend_title="", height=300, yaxis=dict(autorange="reversed"))
fig.show()

### Pregunta 5 · ¿Cuántas facturas emitió cada sede?
Cada fila de la base es una **línea** de factura. Contar `id_factura` cuenta líneas; para facturas necesitas el recuento distinto (`nunique`). Y fíjate: los recuentos distintos de cada fila **no se suman** para obtener el total.

In [ ]:
p5 = validas.pivot_table(index="ciudad", values="id_factura", aggfunc=["count", "nunique"])
p5.columns = ["líneas (count)", "facturas (nunique)"]
p5["productos por factura"] = p5["líneas (count)"] / p5["facturas (nunique)"]
print("Facturas en toda la empresa:", validas["id_factura"].nunique(),
      "| suma de la columna facturas:", p5["facturas (nunique)"].sum())
p5.sort_values("facturas (nunique)", ascending=False).style.format({"productos por factura": "{:.2f}"})

Aquí la suma coincide con el total porque cada factura pertenece a una sola sede. Repite el ejercicio con `index="categoria"` y verás que ya no: una factura puede tener productos de varias líneas.

### Pregunta 6 · ¿En qué meses y ciudades se concentra la venta?
Una tabla dinámica de dos dimensiones se lee mejor como **mapa de calor**: el color hace visible la estacionalidad.

In [ ]:
p6 = (validas[validas["anio"] == 2025]
      .pivot_table(index="ciudad", columns="mes", values="subtotal", aggfunc="sum")
      .loc[p1.index])
fig = px.imshow(p6 / 1e6, x=MESES, y=p6.index, aspect="auto", text_auto=".0f",
                color_continuous_scale=["#EEF2FF", "#9AA8F0", "#4A5AD8", "#00006A"],
                title="Venta neta 2025 por ciudad y mes (millones de pesos)")
fig.update_layout(coloraxis_colorbar_title="M$", xaxis_title="", yaxis_title="", height=380)
fig.show()

## Parte C · Real contra meta
Las metas viven en otra tabla, a nivel sede × año × mes. Para compararlas, resume la venta **al mismo nivel** y luego une. Unir la meta a cada línea de venta y sumar repetiría la meta miles de veces.

In [ ]:
real = (validas.groupby(["id_sede", "anio", "mes"], as_index=False)["subtotal"].sum())
comp = metas.merge(real, on=["id_sede", "anio", "mes"], how="left", validate="one_to_one")
anual = (comp[comp["anio"] == 2025].groupby("id_sede")[["meta_subtotal", "subtotal"]].sum()
         .join(sedes.set_index("id_sede")["ciudad"]))
anual["cumplimiento"] = anual["subtotal"] / anual["meta_subtotal"]
anual = anual.sort_values("cumplimiento", ascending=False)
anual.style.format({"meta_subtotal": pesos, "subtotal": pesos, "cumplimiento": "{:.1%}"})

In [ ]:
datos = anual.sort_values("cumplimiento")
colores = [TEAL if v >= 1 else MAGENTA for v in datos["cumplimiento"]]
fig = go.Figure(go.Bar(x=datos["cumplimiento"], y=datos["ciudad"], orientation="h", marker_color=colores,
                       text=[f"{v:.1%}".replace(".", ",") for v in datos["cumplimiento"]], textposition="outside",
                       cliponaxis=False))
fig.add_vline(x=1, line_dash="dot", line_color="#11153A", annotation_text="Meta 100 %", annotation_position="top")
fig.update_layout(title="Cumplimiento de la meta anual 2025 por sede", xaxis_tickformat=".0%",
                  xaxis_range=[0.8, max(1.15, datos["cumplimiento"].max() + 0.05)], height=360, yaxis_title="")
fig.show()

Las barras en teal cumplieron; las magenta no. El color de estado va siempre con el número escrito, para que el gráfico se entienda también impreso en blanco y negro.

## Parte D · Del resultado al informe
Una tabla dinámica no es un informe. El informe es la frase que alguien puede leer en diez segundos. Usa esta estructura:

1. **Hallazgo con cifra:** qué pasó y cuánto.
2. **Comparación:** contra qué (año anterior, meta, otra sede).
3. **Pregunta siguiente:** qué hay que investigar para decidir.

In [ ]:
mejor, peor = anual.iloc[0], anual.iloc[-1]
print(f"1. En 2025 la empresa vendió {millones(validas.loc[validas.anio == 2025, 'subtotal'].sum())} netos, "
      f"{porc(p2[2025].sum() / p2[2024].sum() - 1, signo=True)} frente a 2024.")
print(f"2. {mejor['ciudad']} superó su meta ({porc(mejor['cumplimiento'])}) y {peor['ciudad']} quedó en {porc(peor['cumplimiento'])}.")
print(f"3. Tecnología concentra la venta pero deja el menor margen ({porc(p3.loc['Tecnología', 'margen % correcto'])}): "
      "¿conviene revisar su política de descuentos?")

## Retos

### Reto 1
¿Qué participación tuvo el **e-commerce** en la venta neta de **2025**? Escríbela en porcentaje con un decimal (por ejemplo, `18.4`).

*Pista:* Usa la tabla `p4` o calcula tú mismo la suma del canal entre la suma total de 2025.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_1
respuesta_1 = None

In [ ]:
# @title Comprobar reto 1
comprobar(respuesta_1, round(p4.loc[2025, 'E-commerce'] * 100, 1), tolerancia=0.05, unidad=' %')

In [ ]:
# @title Ver una solución (inténtalo primero)
v25 = validas[validas.anio == 2025]
respuesta_1 = round(v25.loc[v25.canal == 'E-commerce', 'subtotal'].sum() / v25['subtotal'].sum() * 100, 1)
print(respuesta_1)

### Reto 2
¿Qué ciudad tuvo el **mayor cumplimiento** de meta en 2025? Escribe el nombre como texto, por ejemplo `'Cali'`.

*Pista:* Ordena `anual` por cumplimiento.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_2
respuesta_2 = None

In [ ]:
# @title Comprobar reto 2
comprobar(respuesta_2, anual.sort_values('cumplimiento').iloc[-1]['ciudad'])

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_2 = anual.sort_values('cumplimiento').iloc[-1]['ciudad']
print(respuesta_2)

### Reto 3
¿Cuál fue el margen bruto % de **Tecnología** en **2025**, bien calculado? Escríbelo con un decimal (por ejemplo, `11.2`).

*Pista:* Σ margen_bruto ÷ Σ subtotal, solo 2025, solo Tecnología, sin anuladas.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_3
respuesta_3 = None

In [ ]:
# @title Comprobar reto 3
t = validas[(validas.anio == 2025) & (validas.categoria == 'Tecnología')]
comprobar(respuesta_3, round(t['margen_bruto'].sum() / t['subtotal'].sum() * 100, 1), tolerancia=0.05, unidad=' %')

In [ ]:
# @title Ver una solución (inténtalo primero)
t = validas[(validas.anio == 2025) & (validas.categoria == 'Tecnología')]
respuesta_3 = round(t['margen_bruto'].sum() / t['subtotal'].sum() * 100, 1)
print(respuesta_3)

## Apóyate en IA
La IA es buena para convertir una pregunta de negocio en la configuración de una tabla dinámica, y para revisar si tu interpretación se sostiene:

```text
Tengo un DataFrame de pandas llamado validas con estas columnas:
fecha, id_factura, ciudad, categoria, segmento, canal, subtotal, margen_bruto, estado, anio, mes.
Cada fila es una línea de factura y ya excluí las anuladas.
Pregunta de negocio: "[escribe tu pregunta]".
1. Dime qué va en index, columns, values y aggfunc de pivot_table, y por qué.
2. Señala qué error de agregación debo evitar (conteo de líneas vs facturas, promedio de porcentajes).
3. Propón el gráfico más adecuado y por qué.
No inventes resultados: solo dame el código y el razonamiento.
```

> **Verifica siempre.** Ejecuta el código que te proponga sobre un caso cuyo resultado ya conoces. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero la pregunta, después la tabla dinámica.
* Filtra lo que no es venta. Cuenta facturas con recuento distinto. Calcula porcentajes como razón de sumas.
* Resume al mismo nivel antes de comparar tablas con distinto detalle (venta contra meta).
* Cierra con una frase que tenga cifra, comparación y siguiente pregunta.

**Siguiente:** vuelve al [tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#ruta) para ver la ruta completa de las 12 claves.

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>